In [1]:
import pandas as pd
import numpy as np
import os

# Configuration: Target parameters
TARGET_DT = 0.02  # Target time interval in seconds
TARGET_STEPS = 5000  # Number of steps to keep after downsampling

# List of files to process
file_paths = [
    "/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/SBSP/low_delay.csv"
]

def clean_trajectory_data(file_path):
    """
    Reads a CSV file, downsamples it to TARGET_DT, truncates to TARGET_STEPS,
    and saves the result.
    """
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return

    try:
        # 1. Load Data
        df = pd.read_csv(file_path)
        
        # Ensure data is sorted by time
        if 'time' in df.columns:
            df = df.sort_values('time').reset_index(drop=True)
            
            # 2. Calculate current sampling rate
            # We take the median of differences to avoid outliers/noise
            dt_original = df['time'].diff().median()
            
            if np.isnan(dt_original) or dt_original <= 0:
                print(f"Warning: Could not determine valid sampling rate for {os.path.basename(file_path)}. Assuming step=1.")
                step_size = 1
            else:
                # Calculate step size (e.g., 0.02 / 0.005 = 4)
                step_size = int(np.round(TARGET_DT / dt_original))
                if step_size < 1: 
                    step_size = 1
                
                print(f"Processing {os.path.basename(file_path)}:")
                print(f"  - Original dt: {dt_original:.4f}s")
                print(f"  - Downsampling factor: {step_size} (Target: {TARGET_DT}s)")
        else:
            print(f"Warning: 'time' column missing in {os.path.basename(file_path)}. Skipping temporal downsampling.")
            step_size = 1

        # 3. Downsample
        df_downsampled = df.iloc[::step_size].reset_index(drop=True)

        # 4. Truncate to target steps
        df_cleaned = df_downsampled.head(TARGET_STEPS)
        
        # Verify final shape
        print(f"  - Final shape: {df_cleaned.shape} (Rows, Cols)")

        # 5. Save Output
        # Create output filename: "filename_cleaned.csv"
        file_dir, file_name = os.path.split(file_path)
        name_root, ext = os.path.splitext(file_name)
        output_path = os.path.join(file_dir, f"{name_root}_cleaned{ext}")
        
        df_cleaned.to_csv(output_path, index=False)
        print(f"  - Saved to: {output_path}\n")

    except Exception as e:
        print(f"An error occurred while processing {file_path}: {e}\n")

if __name__ == "__main__":
    print("Starting Data Cleaning Process...\n")
    for path in file_paths:
        clean_trajectory_data(path)
    print("Processing Complete.")

Starting Data Cleaning Process...

Processing low_delay.csv:
  - Original dt: 0.0200s
  - Downsampling factor: 1 (Target: 0.02s)
  - Final shape: (5000, 29) (Rows, Cols)
  - Saved to: /media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/SBSP/low_delay_cleaned.csv

Processing Complete.
